# Portfolio Assessment4 (Task 1)
## Duc Thuan Tran 104330455

In [1]:
import os
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd   
from PIL import Image
import cv2

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import ResNet50

# Task 1: Develop CNN and ResNet50

## Data Preparation

In [2]:
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

corrosion_dir = 'Corrosion'   
train_dir = 'train_directory'  
validation_dir = 'validation_directory'   
test_dir = 'test_set'   

def is_valid_image(img_path):
    try:
        with Image.open(img_path) as img:
            img.verify()
        return True
    except Exception as e:
        print(f"Invalid image detected: {img_path}, Error: {str(e)}")
        return False

def get_valid_images(directory):
    valid_images = []
    for img in os.listdir(directory):
        img_path = os.path.join(directory, img)
        if is_valid_image(img_path):
            valid_images.append(img)
    return valid_images

if os.path.exists(test_dir):
    shutil.rmtree(test_dir)

for directory in [train_dir, validation_dir, test_dir]:
    os.makedirs(os.path.join(directory, 'new_rust'), exist_ok=True)
    os.makedirs(os.path.join(directory, 'new_no_rust'), exist_ok=True)

rust_images = get_valid_images(os.path.join(corrosion_dir, 'new_rust'))
no_rust_images = get_valid_images(os.path.join(corrosion_dir, 'new_no_rust'))

print(f"Found {len(rust_images)} valid rust images")
print(f"Found {len(no_rust_images)} valid no-rust images")

if len(rust_images) < 10 or len(no_rust_images) < 10:
    raise ValueError("Not enough valid images for test set. Need at least 10 of each class.")

selected_rust_test = random.sample(rust_images, 10)
selected_no_rust_test = random.sample(no_rust_images, 10)

for img in selected_rust_test:
    src = os.path.join(corrosion_dir, 'new_rust', img)
    dst = os.path.join(test_dir, 'new_rust', img)
    shutil.copy(src, dst)
    print(f"Copied rust image to test set: {img}")

for img in selected_no_rust_test:
    src = os.path.join(corrosion_dir, 'new_no_rust', img)
    dst = os.path.join(test_dir, 'new_no_rust', img)
    shutil.copy(src, dst)
    print(f"Copied no-rust image to test set: {img}")

rust_test_count = len(os.listdir(os.path.join(test_dir, 'new_rust')))
no_rust_test_count = len(os.listdir(os.path.join(test_dir, 'new_no_rust')))
print(f"Test set contains: {rust_test_count} rust images and {no_rust_test_count} no-rust images")

test_files = []
for img in os.listdir(os.path.join(test_dir, 'new_rust')):
    if is_valid_image(os.path.join(test_dir, 'new_rust', img)):
        test_files.append(os.path.join('new_rust', img))

for img in os.listdir(os.path.join(test_dir, 'new_no_rust')):
    if is_valid_image(os.path.join(test_dir, 'new_no_rust', img)):
        test_files.append(os.path.join('new_no_rust', img))

print(f"Final valid test files: {len(test_files)}")

remaining_rust = [img for img in rust_images if img not in selected_rust_test]
remaining_no_rust = [img for img in no_rust_images if img not in selected_no_rust_test]

validation_split = 0.2
n_val_rust = int(len(remaining_rust) * validation_split)
n_val_no_rust = int(len(remaining_no_rust) * validation_split)

selected_rust_val = random.sample(remaining_rust, n_val_rust)
selected_no_rust_val = random.sample(remaining_no_rust, n_val_no_rust)

selected_rust_train = [img for img in remaining_rust if img not in selected_rust_val]
selected_no_rust_train = [img for img in remaining_no_rust if img not in selected_no_rust_val]

for img in selected_rust_val:
    shutil.copy(
        os.path.join(corrosion_dir, 'new_rust', img),
        os.path.join(validation_dir, 'new_rust', img)
    )

for img in selected_no_rust_val:
    shutil.copy(
        os.path.join(corrosion_dir, 'new_no_rust', img),
        os.path.join(validation_dir, 'new_no_rust', img)
    )

for img in selected_rust_train:
    shutil.copy(
        os.path.join(corrosion_dir, 'new_rust', img),
        os.path.join(train_dir, 'new_rust', img)
    )

for img in selected_no_rust_train:
    shutil.copy(
        os.path.join(corrosion_dir, 'new_no_rust', img),
        os.path.join(train_dir, 'new_no_rust', img)
    )

print(f"Split remaining data: {len(selected_rust_train)} rust and {len(selected_no_rust_train)} no-rust images for training")
print(f"Split remaining data: {len(selected_rust_val)} rust and {len(selected_no_rust_val)} no-rust images for validation")

Invalid image detected: Corrosion/new_rust/.DS_Store, Error: cannot identify image file '/Users/thuanduc/Documents/swinuni/COS40007/Portfolio Assessment 4 /task_1/Corrosion/new_rust/.DS_Store'
Invalid image detected: Corrosion/new_no_rust/.DS_Store, Error: cannot identify image file '/Users/thuanduc/Documents/swinuni/COS40007/Portfolio Assessment 4 /task_1/Corrosion/new_no_rust/.DS_Store'
Found 46 valid rust images
Found 46 valid no-rust images
Copied rust image to test set: 004_i1dmciub.oop.jpg
Copied rust image to test set: OR-07562A_7.JPG
Copied rust image to test set: 002_q40jac40.sd4.jpg
Copied rust image to test set: 007_2pzsqkji.l3n.jpg
Copied rust image to test set: 007_nup2ctfg.0zj.jpg
Copied rust image to test set: 009_kzubx14e.h55.jpg
Copied rust image to test set: 008_oiot10cn.acz.jpg
Copied rust image to test set: 002_cql5orz2.p0y.jpg
Copied rust image to test set: 012_vau2k0co.2q3.jpg
Copied rust image to test set: 030_2vsy3son.gmu.jpg
Copied no-rust image to test set: 00

## Data Augmentation

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.2,
    horizontal_flip=True, 
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=16,  
    class_mode='binary',
    classes=['new_no_rust', 'new_rust'],
    shuffle=True
)

validation_generator = test_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),
    batch_size=16,  
    class_mode='binary',
    classes=['new_no_rust', 'new_rust']
)

test_generator = test_datagen.flow_from_directory(
    test_dir, 
    target_size=(150, 150),
    batch_size=1,  
    class_mode='binary',
    classes=['new_no_rust', 'new_rust'],
    shuffle=False   
)

print("Class indices:", train_generator.class_indices)

Found 82 images belonging to 2 classes.
Found 27 images belonging to 2 classes.
Found 20 images belonging to 2 classes.
Class indices: {'new_no_rust': 0, 'new_rust': 1}


# CNN Model 

In [4]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(150, 150, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Calculate correct steps_per_epoch
steps_per_epoch = len(train_generator)
validation_steps = len(validation_generator)

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=validation_steps
)

model.save('simple_cnn_model.h5')

validation_evaluation = model.evaluate(validation_generator)
print(f"Validation Loss: {validation_evaluation[0]:.4f}, Validation Accuracy: {validation_evaluation[1]:.4f}")

Epoch 1/10


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step - accuracy: 0.3584 - loss: 1.9319 - val_accuracy: 0.5185 - val_loss: 0.6921
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - accuracy: 0.5315 - loss: 0.7364 - val_accuracy: 0.5926 - val_loss: 0.6869
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 112ms/step - accuracy: 0.5454 - loss: 0.6903 - val_accuracy: 0.4815 - val_loss: 0.6907
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - accuracy: 0.5889 - loss: 0.6854 - val_accuracy: 0.5926 - val_loss: 0.6757
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - accuracy: 0.6355 - loss: 0.6744 - val_accuracy: 0.4815 - val_loss: 0.6896
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.4610 - loss: 0.6909 - val_accuracy: 0.4815 - val_loss: 0.6682
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - accuracy: 0.6194 - loss: 0.6405 - val_accuracy: 0.7778 - val_loss: 0.5994
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 113ms/step - accuracy: 0.7808 - loss: 0.5427 - val_accuracy: 0.6667 - val_loss: 0.6322
Epo

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8140 - loss: 0.5515
Validation Loss: 0.5597, Validation Accuracy: 0.8148


# ResNet50 Model

In [5]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(150, 150, 3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

resnet_model = Model(inputs=base_model.input, outputs=predictions)

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),   
    loss='binary_crossentropy',
    metrics=['accuracy']
)

resnet_history = resnet_model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,   
    epochs=10,
    validation_data=validation_generator,
    validation_steps=validation_steps
)

resnet_model.save('resnet50_model.h5')

validation_evaluation = resnet_model.evaluate(validation_generator)
print(f"ResNet50 Validation Loss: {validation_evaluation[0]:.4f}, Validation Accuracy: {validation_evaluation[1]:.4f}")

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 369ms/step - accuracy: 0.5775 - loss: 0.7115 - val_accuracy: 0.4815 - val_loss: 0.6921
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 215ms/step - accuracy: 0.4450 - loss: 0.7506 - val_accuracy: 0.5185 - val_loss: 0.6925
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 214ms/step - accuracy: 0.5240 - loss: 0.7585 - val_accuracy: 0.5185 - val_loss: 0.6907
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 214ms/step - accuracy: 0.4888 - loss: 0.7630 - val_accuracy: 0.5185 - val_loss: 0.6900
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 216ms/step - accuracy: 0.3564 - loss: 0.8205 - val_accuracy: 0.4815 - val_loss: 0.7052
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 239ms/step - accuracy: 0.5474 - loss: 0.7062 - val_accuracy: 0.4815 - val_loss: 0.7098
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 223ms/step - accuracy: 0.5674 - loss: 0.6709 - val_accuracy: 0.4815 - val_loss: 0.6984
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 216ms/step - accuracy: 0.6934 - loss: 0.6525 - val_accuracy: 0.4815 - val_loss:

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.5540 - loss: 0.6834
ResNet50 Validation Loss: 0.6907, Validation Accuracy: 0.5185


## Model Evaluate 

## CNN Results

In [6]:
os.makedirs('cnn_test', exist_ok=True)
os.makedirs('resnet50_test', exist_ok=True)

cnn_model = load_model('simple_cnn_model.h5')

test_generator.reset() 
class_indices = test_generator.class_indices
class_names = {v: k for k, v in class_indices.items()}

filenames = test_generator.filenames
true_classes = test_generator.classes

cnn_predictions = []
for i in range(len(test_generator.filenames)):
    img = test_generator[i][0][0]   
    img = np.expand_dims(img, axis=0)  
    pred = cnn_model.predict(img)
    cnn_predictions.append(pred[0][0])

cnn_predicted_classes = (np.array(cnn_predictions) > 0.5).astype(int)

cnn_results = []
for i, (filename, true_class, prediction) in enumerate(zip(filenames, true_classes, cnn_predictions)):
    img_path = os.path.join('test_set', filename)
    img = plt.imread(img_path)
    
    true_class_name = class_names[true_class]
    pred_class_name = class_names[1] if prediction > 0.5 else class_names[0]
    
    is_correct = (true_class == (1 if prediction > 0.5 else 0))
    title_color = 'green' if is_correct else 'red'
    
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.title(f"True: {true_class_name}, Predicted: {pred_class_name}\nConfidence: {prediction:.4f}", 
              color=title_color)
    plt.axis('off')
    
    output_filename = f'cnn_test/image_{i+1}_{os.path.basename(filename)}'
    plt.savefig(output_filename)
    plt.close()
    
    cnn_results.append({
        'Image': os.path.basename(filename),
        'True Class': true_class_name,
        'Predicted Class': pred_class_name,
        'Confidence': f"{prediction:.4f}",
        'Correct': 'Yes' if is_correct else 'No'
    })

cnn_df = pd.DataFrame(cnn_results)
cnn_df.to_csv('cnn_results.csv', index=False)

cnn_accuracy = np.mean(true_classes == cnn_predicted_classes)
print(f"CNN Test Accuracy: {cnn_accuracy:.4f} ({int(cnn_accuracy * len(true_classes))}/{len(true_classes)} correct)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
CNN Test Accuracy: 0.9000 (18/20 correct)


## ResNet50 Results

In [7]:
resnet_model = load_model('resnet50_model.h5')

test_generator.reset()  

resnet_predictions = []
for i in range(len(test_generator.filenames)):
    img = test_generator[i][0][0]  
    img = np.expand_dims(img, axis=0)   
    pred = resnet_model.predict(img)
    resnet_predictions.append(pred[0][0])

resnet_predicted_classes = (np.array(resnet_predictions) > 0.5).astype(int)

resnet_results = []
for i, (filename, true_class, prediction) in enumerate(zip(filenames, true_classes, resnet_predictions)):
    img_path = os.path.join('test_set', filename)
    img = plt.imread(img_path)
    
    true_class_name = class_names[true_class]
    pred_class_name = class_names[1] if prediction > 0.5 else class_names[0]
    
    is_correct = (true_class == (1 if prediction > 0.5 else 0))
    title_color = 'green' if is_correct else 'red'
    
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.title(f"True: {true_class_name}, Predicted: {pred_class_name}\nConfidence: {prediction:.4f}", 
              color=title_color)
    plt.axis('off')
    
    output_filename = f'resnet50_test/image_{i+1}_{os.path.basename(filename)}'
    plt.savefig(output_filename)
    plt.close()
    
    resnet_results.append({
        'Image': os.path.basename(filename),
        'True Class': true_class_name,
        'Predicted Class': pred_class_name,
        'Confidence': f"{prediction:.4f}",
        'Correct': 'Yes' if is_correct else 'No'
    })

resnet_df = pd.DataFrame(resnet_results)
resnet_df.to_csv('resnet50_results.csv', index=False)

resnet_accuracy = np.mean(true_classes == resnet_predicted_classes)
print(f"ResNet50 Test Accuracy: {resnet_accuracy:.4f} ({int(resnet_accuracy * len(true_classes))}/{len(true_classes)} correct)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 522ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
ResNet50 Test Accuracy: 0.5000 (10/20 correct)
